# NeuroTrain Lab — Notebook 4: Training and Overfitting

**Topic:** how to organize a real training run end to end — a
train/validation/test split, scaling without information leakage,
an honest baseline, `Dropout` and `EarlyStopping` — and how to spot
overfitting by reading the learning curves.

> Last of 4 notebooks. You already know the perceptron (NB1), loss
> and backprop (NB2), and optimizers (NB3). Here we put it all
> together to train a real model on a real problem and learn to
> **control** overfitting, not just observe it.

## 🎯 What you'll learn in this notebook

By the end you should be able to explain, without memorized formulas:

1. Why we split into three sets (train/validation/test), not two.
2. Why scaling is fit only on train.
3. What `Dropout` and `EarlyStopping` do, and why they're combined.
4. How to read `loss` and `val_loss` to diagnose overfitting.
5. Why we always compare against a simpler baseline.

**Mental map:** `real data → split → scaling → baseline → MLP → EarlyStopping+Dropout → fit() → curves → test vs baseline → A/B experiment`

In [ ]:
from pathlib import Path
import json
import math
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate
from neurotrain.config import TrainingConfig
from neurotrain.data import load_dataset, prepare_data
from neurotrain.evaluation import classification_metrics
from neurotrain.modeling import train_dense_classifier
from neurotrain.visualization import (
    plot_confusion,
    plot_roc,
    plot_training_history,
    plot_training_history_comparison,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

## 1. Quick dataset audit

You already glanced at this CSV in Notebook 1. Before training we
confirm its minimal contract with `assert`: shape, no missing
values, and the two expected labels. This is the last time we do it
"by hand" — the rest of the project uses
`neurotrain.data.load_dataset()`, which runs exactly these checks.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv"
df = pd.read_csv(DATA_PATH)

assert df.shape == (569, 31)
assert not df.isna().any().any()
assert set(df["diagnosis"].unique()) == {"B", "M"}

class_counts = df["diagnosis"].value_counts().rename(index={"B": "Benign", "M": "Malignant"})
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
display(class_counts.to_frame("records"))

## 2. Split X and y

The positive class is explicitly defined as **malignant = 1** — so
"sensitivity" unambiguously means "proportion of true malignant
cases we detect."

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"].eq("M").astype("int8")

print("X shape:", X.shape, "| y shape:", y.shape)

## 3. Creating train, validation, and test

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — Why three sets, not two?</b><br><br>
`train` is the practice set the network adjusts its weights on.
`validation` is a mock exam: the network never learns from it
directly, but **we** use it to decide architecture, dropout,
patience, or threshold. `test` is the final exam, opened only
**once**, at the very end. If you use test to decide anything, it
stops measuring what it's supposed to measure — it just measures
how well you overfit to the exam.
</div>

We'll use roughly 70% / 15% / 15%. `stratify=y` keeps a similar
proportion of benign and malignant cases in every split.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    {
        "records": [len(X_train), len(X_val), len(X_test)],
        "% malignant": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
display(split_summary.style.format({"% malignant": "{:.1%}"}))

## 4. Scaling without information leakage

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ TYPICAL MISTAKE — The most common mistake: scaling before splitting</b><br><br>
If you fit `StandardScaler` on the **whole** dataset and split
afterwards, train's mean and standard deviation already "saw"
validation and test examples. It's a subtle leak: the model
doesn't copy answers, but its preprocessing already benefited from
the final exam. The rule is always: `fit_transform` only on train,
`transform` on the rest.
</div>

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

print("Approx. train mean:", X_train_scaled.mean(axis=0)[:3].round(5))
print("Shape the network will receive:", X_train_scaled.shape)

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🌱 You just planted the first seed of your neural network.</div>

## 5. Creating an honest baseline

A logistic regression answers the same question and is much
simpler. If it performs as well as or better than the network, the
professional conclusion isn't "the ANN failed": it's "the extra
complexity wasn't justified by this data."

In [ ]:
baseline = LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)
baseline.fit(X_train_scaled, y_train)

baseline_probabilities = baseline.predict_proba(X_test_scaled)[:, 1]
print("Baseline ROC-AUC:", round(roc_auc_score(y_test, baseline_probabilities), 3))

## 6. Epochs, batches, and iterations

You already used the "batch"/"step" vocabulary in Notebook 3 when
discussing optimizers. Here we just ground it in concrete numbers
for **this** split: given a fixed-size `X_train_scaled` and a
`batch_size`, how many weight updates happen per epoch?

In [ ]:
BATCH_SIZE = 32
EPOCHS = 200

### ✏️ Exercise

Complete the updates-per-epoch calculation **using the actual size
of `X_train_scaled`**, not a fixed number. Remember: one update
happens per batch processed, and the epoch's last batch can be
smaller (that's why we round up with `math.ceil`).

In [ ]:
updates_per_epoch = math.ceil(✏️✏️✏️)
max_updates = updates_per_epoch * EPOCHS

print("Updates per epoch:", updates_per_epoch)
print("Max updates (if all epochs run):", max_updates)

<details>
<summary><b>Show solution</b></summary>

```python
updates_per_epoch = math.ceil(len(X_train_scaled) / BATCH_SIZE)
max_updates = updates_per_epoch * EPOCHS

print("Updates per epoch:", updates_per_epoch)
print("Max updates (if all epochs run):", max_updates)
```

</details>

## 7. Building the network

`30 features → Dense(32, ReLU) → Dropout(0.30) → Dense(16, ReLU) → Dense(1, Sigmoid)`

Why 32 and 16 neurons (hyperparameters to validate, not formulas
derived from the feature count) was already covered in Notebook 1.
The only new part here is the output layer: a single neuron with a
**Sigmoid** activation produces the probability we use as a
prediction.

### ✏️ Exercise

Complete the output layer's activation. Hint: we need a single
number between 0 and 1, interpretable as the probability of
"malignant" — the same function you used in Notebook 1 for the
binary output layer.

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="✏️✏️✏️"),
    ],
    name="neurotrain_mlp",
)
model.summary()

<details>
<summary><b>Show solution</b></summary>

```python
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="neurotrain_mlp",
)
model.summary()
```

</details>

## 8. Compiling: optimizer, loss, and metrics

`compile()` doesn't train yet, it only configures the rules. You
studied the optimizer (Adam) in depth in Notebook 3; you studied
the loss (binary cross-entropy) in depth in Notebook 2. Here we
just wire them together with metrics we observe but that never
replace the loss.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="sensitivity"),
    ],
)

## 9. EarlyStopping and Dropout

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — Dropout: forcing redundancy</b><br><br>
`Dropout(0.30)` randomly switches off 30% of that layer's neurons
on **every training step**. It forces the network to not always
rely on the same internal paths — like studying without
memorizing the exact order of the questions. At inference
(real prediction) no neuron is switched off.
</div>

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 KEY CONCEPT — EarlyStopping: stopping at just the right moment</b><br><br>
Watches `val_loss` epoch by epoch. If it doesn't improve for
`patience` consecutive epochs, training stops.
`restore_best_weights=True` recovers the weights from the
**best** observed epoch, not the last one — so a late-training
regression doesn't become the final result.
</div>

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ LIKELY QUESTION — Does validation train the network too?</b><br><br>
No. `validation_data` is evaluated at the end of every epoch only
to **measure**; its examples never participate in the gradient
computation or update weights. That's why it can be used to decide
when to stop without "cheating."
</div>

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=1,
)

## 10. Training with `fit()`

In every batch, the four steps you already dissected in Notebooks
2 and 3 happen in one line: forward pass → loss → backpropagation
→ optimizer step.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=0,
)

print(f"Epochs run: {len(history.history['loss'])} of {EPOCHS}")

## 11. Reading the learning curves

- If `loss` and `val_loss` go down together, the network is
  learning patterns that generalize.
- If `loss` keeps dropping while `val_loss` rises, the network is
  **memorizing** train: overfitting.
- If both stay high, there may be underfitting, too few epochs, or
  an unsuitable configuration.

Don't look only at `accuracy`: with imbalanced classes it can look
good even while the model fails exactly the class that matters.

In [ ]:
history_dict = {key: list(values) for key, values in history.history.items()}
fig = plot_training_history(history_dict, lang="en")
plt.show()

## 12. Evaluating once on test, against the baseline

Initial threshold of 0.50: probability ≥ 0.50 becomes "malignant"
(1). Sensitivity, specificity, precision, and ROC-AUC are computed
by `classification_metrics` — the same logic you'd write by hand,
packaged so it isn't repeated in every notebook.

In [ ]:
THRESHOLD = 0.50
probabilities = model.predict(X_test_scaled, verbose=0).ravel()

ann_metrics = classification_metrics(y_test, probabilities, THRESHOLD)
baseline_metrics = classification_metrics(y_test, baseline_probabilities, THRESHOLD)

comparison = pd.DataFrame(
    {"Network (MLP)": ann_metrics, "Baseline (LogReg)": baseline_metrics}
).loc[["accuracy", "roc_auc", "precision", "sensitivity", "specificity", "f1"]]
display(comparison.style.format("{:.3f}"))

In [ ]:
fig_confusion = plot_confusion(y_test, probabilities, threshold=THRESHOLD, lang="en")
plt.show()

fig_roc = plot_roc(y_test, probabilities, lang="en")
plt.show()

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 REMEMBER THIS — Beating the baseline isn't optional to justify the network</b><br><br>
If the network doesn't clearly beat the logistic regression, the
correct professional call is usually to **use the simpler model**:
it's cheaper to train, easier to explain, and less prone to
overfitting with limited data.
</div>

## 13. Guided overfitting experiment (A vs B)

Instead of re-editing the cells above (risking losing your
reference run), we'll launch **two independent configurations**
that coexist, using `TrainingConfig`:

| Variant | Layers | Dropout | EarlyStopping | Hypothesis |
|---|---:|---:|---:|---|
| A | 128 → 64 | 0.0 | No | Train will improve; validation might get worse |
| B | 32 → 16 | 0.30 | Yes | Less capacity to memorize, earlier stopping |

Variant B is actually the same architecture you just trained by
hand in Sections 7-10 — here we reproduce it via `TrainingConfig`
so it sits side by side with A.

Before running, **write your prediction** in your own text cell:
which one do you think will have the lower minimum `val_loss`?

`load_dataset()` and `prepare_data()` are the **same logic** you
wrote by hand in Sections 1, 3, and 4 (audit, stratified split,
`StandardScaler` fit only on train) — packaged so a real project
doesn't repeat it in every experiment.

In [ ]:
frame = load_dataset()
data = prepare_data(frame, random_state=RANDOM_STATE)
print("Train:", data.X_train.shape, "| Val:", data.X_val.shape, "| Test:", data.X_test.shape)

### ✏️ Exercise

Complete `config_a` following the table's hypothesis: no Dropout
and no EarlyStopping, so the network can memorize train freely
over the 200 epochs.

In [ ]:
config_a = TrainingConfig(
    hidden_units=(128, 64),
    dropout_rate=✏️✏️✏️,
    use_early_stopping=✏️✏️✏️,
    epochs=200,
    batch_size=32,
    random_state=RANDOM_STATE,
)
model_a, history_a = train_dense_classifier(data, config_a)
print("Epochs run (A):", len(history_a["loss"]))

<details>
<summary><b>Show solution</b></summary>

```python
config_a = TrainingConfig(
    hidden_units=(128, 64),
    dropout_rate=0.0,
    use_early_stopping=False,
    epochs=200,
    batch_size=32,
    random_state=RANDOM_STATE,
)
model_a, history_a = train_dense_classifier(data, config_a)
print("Epochs run (A):", len(history_a["loss"]))
```

</details>

In [ ]:
config_b = TrainingConfig(
    hidden_units=(32, 16),
    dropout_rate=0.30,
    use_early_stopping=True,
    patience=12,
    epochs=200,
    batch_size=32,
    random_state=RANDOM_STATE,
)
model_b, history_b = train_dense_classifier(data, config_b)
print("Epochs run (B):", len(history_b["loss"]))

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🏁 Final stretch before the end of this notebook.</div>

In [ ]:
fig_comparison = plot_training_history_comparison(
    history_a,
    history_b,
    "A: 128->64, no regularization",
    "B: 32->16, with Dropout+EarlyStopping",
    lang="en",
)
plt.show()

Answer with the plot in front of you:

1. At which epoch was `val_loss` minimal for each variant?
2. How far apart were `loss` and `val_loss` in A? And in B?
3. Did the bigger network (A) improve the test result, or only train?
4. Did either variant justify being more complex than the Section 5 baseline?

## 14. Saving the model and preprocessing

A model without its scaler doesn't reproduce the same pipeline: we
save both, plus minimal metadata for the reference experiment
(variant B, which we evaluated on test in Section 12).

In [ ]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

model.save(ARTIFACTS_DIR / "neurotrain_model.keras")
joblib.dump(scaler, ARTIFACTS_DIR / "scaler.joblib")

metadata = {
    "dataset": "UCI Breast Cancer Wisconsin Diagnostic",
    "positive_class": "M = 1",
    "feature_names": X.columns.tolist(),
    "threshold": THRESHOLD,
    "epochs_executed": len(history.history["loss"]),
    "test_metrics": ann_metrics,
    "intended_use": "educational demonstration only",
}
(ARTIFACTS_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Artifacts saved to:", ARTIFACTS_DIR)

## 15. From notebook to product

Everything you did across these 4 notebooks also lives in a
Streamlit app with a **guided journey**: a "Home" page and one
page per topic (Perceptron, Loss & backprop, Optimizers, Training —
this same notebook, summarized visually).

It also has an **"Experiment Mode"**: a lab page where you can
retrain interactively — changing architecture, dropout, epochs,
patience, and threshold — with live per-epoch progress and a panel
that reveals the real code behind each button.

```powershell
streamlit run app.py
```

**Next step:** open the app, reproduce variants A and B from
Experiment Mode, and explain out loud what changed. If you can
justify the result without rereading this notebook, you've
mastered the core of the masterclass.

## 🎯 Self-assessment

Answer without looking back. You don't need perfect phrasing: explain the mechanism in your own words.

**1. Why do we split off a validation set in addition to train and test?**

A. Because the model needs more data to learn
B. To decide architecture, dropout, patience, or threshold without touching test
C. Because test must always be larger than train
D. Validation and test are the same set under a different name

<details>
<summary><b>Show answer</b></summary>

**B.** Validation guides human configuration decisions during development; test is opened only once, at the end, so the measurement isn't contaminated.

</details>

**2. What data should `StandardScaler` be fit on?**

A. The whole dataset, before splitting
B. Only train
C. Only test
D. Train and validation together, but never test

<details>
<summary><b>Show answer</b></summary>

**B.** Fitting the scaler on data outside train leaks final-exam information into preprocessing, even though the model never directly 'sees' those labels.

</details>

**3. What does `restore_best_weights=True` do in `EarlyStopping`?**

A. Resets the weights to random values at the end
B. Keeps the weights from the last epoch, good or bad
C. Recovers the weights from the epoch with the best observed `val_loss`
D. Freezes the first epoch's weights as a reference

<details>
<summary><b>Show answer</b></summary>

**C.** Without this option, the model would keep the last trained epoch's weights, which can be worse than an earlier epoch if it was already regressing.

</details>

**4. In the A/B experiment, variant A (128->64, no Dropout, no EarlyStopping) shows train `loss` dropping a lot while `val_loss` rises past a certain epoch. What's happening?**

A. Underfitting
B. Overfitting: the network memorizes train and stops generalizing
C. A code bug, that combination shouldn't happen
D. The learning rate is too low

<details>
<summary><b>Show answer</b></summary>

**B.** This is the classic overfitting signature: the network keeps reducing error on data it sees, but gets worse on new data because it memorized train details.

</details>

**5. Which optimizer and loss did we use to compile the network in this notebook, and why (per Notebooks 2 and 3)?**

A. Plain SGD and MSE, because they're the simplest
B. Adam and binary cross-entropy, because Adam adapts the learning rate per-parameter and BCE penalizes miscalibrated probabilities in binary classification
C. Momentum and categorical cross-entropy, because there are more than two classes
D. Adam and accuracy, because accuracy is what's actually minimized

<details>
<summary><b>Show answer</b></summary>

**B.** Adam (NB3) combines momentum with per-parameter adaptive learning rates; binary cross-entropy (NB2) is the correct loss for binary classification with a Sigmoid output. Accuracy is a monitoring metric, not the minimized function.

</details>

**6. A model reaches 99% train accuracy and 71% validation accuracy. What's happening, and what would you try first?**

<details>
<summary><b>What a good answer should include</b></summary>

- Names the phenomenon: overfitting (the network memorizes train, doesn't generalize).
- Proposes reducing capacity (fewer neurons/layers) or raising Dropout.
- Proposes enabling or tightening EarlyStopping (lower patience) to stop training past the best-val_loss point.
- Mentions this could also stem from too little train data for the model's complexity.

</details>

**7. Explain to a non-technical stakeholder why we compare the neural network against a plain logistic regression instead of trusting the network outright.**

<details>
<summary><b>What a good answer should include</b></summary>

- Makes clear a complex model isn't automatically better; it has to be proven with data.
- Mentions the baseline is cheaper, faster to train, and easier to explain to others.
- Explains that if the baseline ties or wins, the professional conclusion is to use the simple model, not force the network.

</details>

In [ ]:
celebrate(
    "🎉 Congratulations! You finished all 4 NeuroTrain Lab notebooks 🎉",
    "From the perceptron to training with overfitting under control: you now know "
    "the whole path. Now open the Streamlit app and put your A and B variants to "
    "the test in Experiment Mode.",
)